In [13]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from pathlib import Path

import numpy as np
import pandas as pd

### Config

In [15]:
from config import dir_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

# Following viewing times to be added in reaction time to account for actual accumulation time
min_viewing_time_uw = 0.3
min_viewing_time_ucla = 0.3

### Aggregate Data

### Data is collected from two different set of mice. 
1. UCLA: Big screen, 2 pulse times 
2. UW: smaller screeen, 3 pulse times (check reaction times)

In [16]:
ucla_sessions_trial = list(raw_dir.glob("PSUIM*/*/*_trial.csv"))
ucla_sessions_event = list(raw_dir.glob("PSUIM*/*/*_event.txt"))
ucla_sessions_lick = list(raw_dir.glob("PSUIM*/*/*_lick.txt"))

uw_sessions_trial = list(raw_dir.glob("DBM*/*/*_trial.csv"))
uw_session_licks = list(raw_dir.glob("DBM*/*/*_lick.txt"))

In [17]:
raw_dir


PosixPath('/mnt/data/raw')

In [18]:
# Initialize empty DataFrames
ucla_data = pd.DataFrame()
uw_data = pd.DataFrame()

# Process UCLA sessions
for session in ucla_sessions_trial:
	temp_df = pd.read_csv(session)
	temp_df["subject_id"] = session.stem.split("_")[0]  # Extract subject_id
	temp_df["session_id"] = session.parent.name  # Get the parent folder name as session_id
	ucla_data = pd.concat([ucla_data, temp_df], ignore_index=True)

# Process UW sessions
for session in uw_sessions_trial:
	temp_df = pd.read_csv(session)
	temp_df["subject_id"] = session.stem.split("_")[0]  # Extract subject_id
	temp_df["session_id"] = session.parent.name  # Get the parent folder name as session_id
	uw_data = pd.concat([uw_data, temp_df], ignore_index=True)

In [19]:
# create new data which is combined from UCLA and UW data

# meta data
data_source = ["ucla"] * len(ucla_data) + ["uw"] * len(uw_data)
data_source = pd.Series(data_source, name="data_source")
subject_id = pd.concat([ucla_data["subject_id"], uw_data["subject_id"]], ignore_index=True)
session_id = pd.concat([ucla_data["session_id"], uw_data["session_id"]], ignore_index=True)

# trial counters
idx_valid_trial = pd.concat([ucla_data["TrialNo"], uw_data["idx_valid"]], ignore_index=True)
idx_correction = pd.concat([ucla_data["AttemptNo"], uw_data["idx_correction"]], ignore_index=True)
is_correction_trial = pd.concat([(ucla_data["AttemptNo"] != 0), uw_data["is_correction_trial"]], ignore_index=True)

# trial info
coherence = pd.concat([ucla_data["Coherence"], uw_data["signed_coherence"]], ignore_index=True)
target = pd.concat([np.sign(ucla_data["Coherence"]), uw_data["target"]], ignore_index=True)
choice = pd.concat([ucla_data["Response"], uw_data["choice"]], ignore_index=True)
reaction_time = pd.concat([ucla_data["ReactionTime"], uw_data["response_time"] + min_viewing_time_uw], ignore_index=True)  # adding min viewing time to reaction time to account for stimulus viewing time
outcome = pd.concat([ucla_data["Correct"], uw_data["outcome"]], ignore_index=True)
reward_amount = pd.concat([ucla_data["RewardAmount"], uw_data["trial_reward"]], ignore_index=True)
stimulus_seed = pd.concat([ucla_data["TrialNo"], uw_data["stimulus_seed"]], ignore_index=True)

# timing info
fixation_duration = pd.concat([ucla_data["FixationTime"], uw_data["fixation_duration"]], ignore_index=True)
stimulus_duration = pd.concat([ucla_data["StimulusTime"], uw_data["stimulus_duration"]], ignore_index=True)
reinforcement_duration = pd.concat([ucla_data["StimTimeOut"], uw_data["reinforcement_duration"]], ignore_index=True)
intertrial_duration = pd.concat([ucla_data["InterTrialInterval"], uw_data["delay_duration"] + uw_data["intertrial_duration"]], ignore_index=True)

# pulse info
pulse_onset = pd.concat([ucla_data["BurstTime"], uw_data["pulse_onset"] * 16.67 / 1000], ignore_index=True)
pulse_duration = pd.concat([ucla_data["BurstDur"], uw_data["pulse_duration"] * 16.67 / 1000], ignore_index=True)
pulse_coherence = pd.concat([np.sign(ucla_data["BurstCoh"]) * np.sign(ucla_data["Coherence"]) * 100, uw_data["pulse_coherence"]], ignore_index=True)
is_pulse_trial = pd.concat([ucla_data["BurstCoh"] != 0, uw_data["pulse_coherence"] != uw_data["signed_coherence"]], ignore_index=True)

In [20]:
# crate new dataframe with above series with their respective names
aggregate_data = pd.DataFrame()
aggregate_data["data_source"] = data_source
aggregate_data["subject_id"] = subject_id
aggregate_data["session_id"] = session_id
aggregate_data["idx_valid_trial"] = idx_valid_trial
aggregate_data["idx_correction"] = idx_correction
aggregate_data["is_correction_trial"] = is_correction_trial
aggregate_data["signed_coherence"] = coherence
aggregate_data["target"] = target
aggregate_data["choice"] = choice
aggregate_data["reaction_time"] = reaction_time
aggregate_data["outcome"] = outcome
aggregate_data["reward_amount"] = reward_amount
aggregate_data["fixation_duration"] = fixation_duration
aggregate_data["stimulus_duration"] = stimulus_duration
aggregate_data["reinforcement_duration"] = reinforcement_duration
aggregate_data["intertrial_delay"] = intertrial_duration
aggregate_data["is_pulse_trial"] = is_pulse_trial
aggregate_data["pulse_onset"] = pulse_onset
aggregate_data["pulse_duration"] = pulse_duration
aggregate_data["pulse_coherence"] = pulse_coherence
aggregate_data["stimulus_seed"] = stimulus_seed

In [21]:
# Get number of unique subjects and how many sessions they have
subject_sessions = aggregate_data.groupby("subject_id")["session_id"].nunique().reset_index(name="count")
# remove subjects with less than 2 sessions
subjects_to_remove = subject_sessions[subject_sessions["count"] < 2]["subject_id"]
aggregate_data = aggregate_data[~aggregate_data["subject_id"].isin(subjects_to_remove)]
# Get number of unique subjects and how many sessions they have
subject_sessions = aggregate_data.groupby("subject_id")["session_id"].nunique().reset_index(name="count")
subject_sessions

,subject_id,count
0,DBM11,19
1,DBM12,17
2,DBM13,9
3,DBM14,20
4,DBM15,50
5,DBM16,14
6,DBM5,34
7,DBM6,33
8,DBM8,22
9,PSUIM10,9


In [22]:
# insert a column for the index of the session (subject_id, session_id)
aggregate_data.insert(0, "idx_session", pd.Categorical(aggregate_data[["subject_id", "session_id"]].apply(tuple, axis=1)).codes)

In [23]:
ucla_data.to_csv(Path(raw_dir, "ucla_data.csv"), index=False)
uw_data.to_csv(Path(raw_dir, "uw_data.csv"), index=False)

aggregate_data.to_csv(Path(raw_dir, "aggregate_data.csv"), index=False)